In [ ]:
"""
Q- Find the highest salary from a table 

select employee_name, max(salary) from employee_table

"""

#### The sql execution order

##### From-->join-->where-->groupby-->agg(avg, sum)-->having-->select-->distinct-->orderby-->limit

In [ ]:
## Second highest salary  
"""
## approach 1 by using limit and subquery (do not use this approch in interview)

select salary from (slect salary from employee_table
order by salary desc limit 2) as temp 
order by salary asc limit 1

"""

## approch(2) 
"""
select max(salary) from employee_table  where salary < (
select max(salary) from employee_table)
"""

## approach(3) by using the window function dens_rank()

"""

select salary from 
 (select salary, dense_rank() over(order by salary desc) as d_rnk
 from employee_table) as tem_result
 where d_rank=2;
 

"""


In [ ]:
## Highest salary per department

"""
approach 1 
 select department_name, max(salary) as highest_salary
 from employee_table group by department_name

"""
## approach 2 
"""
 select  salary, employee_name, department_name from
 (select salary, department_name, employee_name, rank() over (partition by department_name order by salary desc) as max_salary 
 FROM employee_table ) tmp
 where max_salary=1

"""


In [ ]:
## second highest salary per department 

"""
## approach1 using group by 

-- second highest salary 
select department_name, max(salary) as salary from employee_table e1
where salary < (
select max(salary) from employee_table d1 where d1.department_name=e1.department_name)
group by department_name

 
##byusing windows function  (approach) 2

select department_name, employee_name, salary from 
(select department_name, employee_name, salary , dense_rank() over(partition by department_name order by salary desc) as drnk
from employee_table) t 
where drnk=2;

"""


In [ ]:
## Find total salary paid per department
"""
## approach 1 

select department, sum(salary) as total_salary  
from employee_table group by department

## approach 2 using window_function  -
select distinct department_name, total_salary from (
select department_name, sum(salary) over(partition by department_name) as total_salary
from department_table) tmp;

Note- why distinct here because (windows function here produced the multiple rows per department 
so we used distinct)

"""

In [ ]:
##	Find departments where avg salary > 60,000

"""  
## approach1  (best approach)
select department_name, avg(salary) as avg_salary
from employee_table
group by department_name 
having avg(salary)>600000   -- we can use having clause 


## approach 2 using window function
select DISTINCT deparment_name, avg_salary from 
  (selet department_name, avg(salary) over (partition by department) as avg_salary  
  from employee_table) tmp 
  where avg_salary>60000


"""

In [ ]:
## Find duplicate employee names
"""
## approach1  ( this approach is good for summary ) count(*) does not ignore nulls

 select employee_name, count(employee_name) as cnt from employee_table
  group by employee_name
  having count(employee_name)>1;  


## by using window function  ( for large table scan )

  select * from (
  select employee_name, count(employee_name) over(partition by employee_name) as cnt 
  from employee_table ) tmp 
  where cnt>1;

"""


## ------------What if interviewer asks:

# “Show me the actual duplicate rows, not just the names”

"""
Answer -  ( for large table scan)
select * from (
 select employee_name
 from employee_table 
 group by employee_name 
 having count(employee_name) >1);

"""

In [ ]:
##Remove duplicate employee records, keeping only one row per name

## approach 1 keep the row with first employee_id (lowest employee id )
"""
with duplicate_rows as (
select *, row_number() over(partion by employee_name) as rn 
from employee_table 
)

delete from  employee_table where employee_id in (
select employee_id from duplicate_rows where
rn>1);


----------approach 2 (keep only last updated rows)

with duplicate_rows (
select *, row_number() over(partition by employee_name order by ts desc) as rn  from employee_table
)
delete from employee_table where employee_id in
(select employee_id from duplicate_rows where rn>1);


"""


In [ ]:
## 10.	Find employees earning more than average salary

"""  
## approach 1 (scaler subquery- the subquery runs only once)

select employee_name from employee_table where salary > (
 select avg(salary) from employee_table 
);

## approach 2 (window function) (without subquery)
select  employee_name, salary from (
select employee_name, salary, avg(salary) over () as avg_salary from employee_table ) t 
where salary> avg_salary; 


"""


In [ ]:
## 11.	Find departments with no employees

"""  both approaches are wrong ,because group by cannot be follow on now rows departments
approach 1 

select department, count(employee_id) as employee_cnt from employee_table
group by department
having count(employee_id)<1;

--approach 2


select department_name, ep_cnt (
select department_name, count(employee_id) over (partition by departemnt_name) as ep_cnt
 from employee_table
 )
 where ep_cnt<1;

"""
# -------------correct way
    #LEFT JOIN keeps all departments

   #Departments with no employees → NULL on employee side
   #Filter NULLs → exactly what we want
"""
## left join 
select d.department_name from department d
left join employe_table e
on d.department_id=e.department_id
where e.employee_id is null;

## approach 2

select d.department_name from department_table d
where not exists (
select 1 from employee_table 
where e.department_id=d.department_id);
"""


In [ ]:
#Find employees count per job role

 #---- approach 1 (common group by approach)
"""
select job_role, count(employee_id) as employee_count 
from employee_table 
group by job_role
""" 



In [ ]:
# Get the latest transaction for each customer. 

"""
select * from (
select *, row_number() over(partition by customer order by transaction_date)  rn
from transaction_fact ) fct
where rn=1;


"""

In [ ]:
#Find top 3 branches per day by total transaction amount.
"""
 ## this approach is wrong because we cannot use having clause with count in this way
select branch_id, sum(transaction_amount) as total_t_amount, timestamp
from transaction_fact 
group by timestamp, branch_id
having count(branch_id)=3

"""

## approach 2 
"""
select * from (
select branch_id, sum(transaction_amount) as total_amount,
datetime(txn_timestamp) as timestamp
dense_rank() over(partition by datetime(txn_timestamp) order by sum(transaction_amount)) drnk
from transaction_fact 
group by datetime(txn_timestamp),branch_id) tmp
where drnk<=3

"""

## with cte
with top_brach as (
    
    select datetime(timestamp) as timestamp, branch_id, sum(total_amount) as total_txn
    from txn_table
    group by datetime(timestamp) , branch_id
)

select * from 
(
    select *, dense_rank() over(partition by timestamp order by total_txn) drnk 
    from top_branch  
) tmp
where drnk<=3;

In [ ]:
#Calculate month-over-month transaction growth per branch.
"""
with monthly as (
   select branch_id, date_trun('month', timestamp) as month,
   sum(transaction_amount) as total_amount
   from txn_fact_table
   group by date_trun('month', timestamp) , branch_id
)
select branch_id, month
  , total_amount, total_amount-lag(month) over(partition by branch_id order by month) as growth
   from monthly
"""

In [ ]:
#Find duplicate transaction_ids.
"""
select transaction_id , count(*)
from transaction_fact 
having count(*)>1;

"""

## keep the latest transaction record

 """
 select * from(
  select transaction_id, timestamp , row_number() over(partition by transaction_id 
                                   order by timestamp desc) as rn
                                   from txn_fact_table) tmp 
    where rn=1;
 """

In [ ]:
#Find customers who transacted on 3 consecutive days.

with customer_txn as (
    select distinct customer,
    date(timestamp) as timestamp
    from transaction_fact
),
consecutive_numbers as (
    select customer, timestamp
    timestamp-row_number() over(partition by customer order by timestamp) as grp
    from customer_txn
)
select customer_id 
from consecutive_numbers 
group by customer_id , grp
having count(*) >=3

In [ ]:
# Calculate running total per customer.

selet * from (select customer_id, txn_timestamp, transaction_amount
sum(total_ammount) over(partiton by customer_id order by timesatmp) as running_total
rows between unbounded preceding and current row
from fact_txn_table) tmp;

In [ ]:
#Find branches where total amount increased compared to previous day.

In [ ]:
# cosecutive days problem give the login date with 3 consecutive days

select * from 
(
    select *, login_date-row_number(order by login date) as grp 
    from login_table
)t 
group by grp 
having count(*)>=3;

In [ ]:
#Delete duplicates (with & without window functions)

""" without window function approach"""

# used the min() approach , only keeping minumum id  - because this is inefficient
delete from employee_table 
where id not in (
select min(id) from employee_table where id is not null group by emp_id or emp_name);

# more better using self join

delete e1 from employee e1 
join employee e2 
on e1.name=e2.name
and e1.id>e2.id;

## with window function 
delete from employee where id in 
(select  id from 
 (
     select id, row_number(over partition by emp_name order by ts desc) t 
     from employee_table
 )t 
 wher rn>1
 );

In [ ]:
from pyspark.sql.functions import *
from pyspark.window import Window
movie_df=spark.read.format('csv') \
       .option('inferSchema',True) \
        .option('header',True) \
        .load('/Volumes/my_data/default/vaibhav/imdb_top_250_movies_clean.csv')
movie_df.limit(10).display()

## find the 
"""" pick the latest record from customer from transaction history y and return the a record"""
wid= Window.partitionBy("customerid").orderBy(col("transactiondate").desc())
movie_df=withColumn("rn",row_number().over(wid)).filter("rn"==1).drop("rn")


## rank salepersion per region
wid2= Window.partitionBy("Item_type").orderBy(col("Item_outlet_sales").desc())
sales_df_new=sales_df.withColumn("rank",rank().over(wid2))

## dense rank to get the top 3 sales per item_type

wid_dens=Window.partitionBy("Item_Type").orderBy(col("Item_outlet_sales").desc())
sales_df_new=sales_df.withColumn("dense_rank",dense_rank().over(wid_dens)).filter("desne_rank<=3")

In [ ]:
# find the difference in daily stock compared to previous day
wind_lag=Window.partitionBy("stockid").orderBy(col("date").desc())
stock_df_lag=stock_df.withColumn("previous_col",lag('price',1).over(wind_lag))
stock_lag_df=stock_df_lag.withColumn("difference",col('previous_col')-col("current_date"))

In [ ]:
## You have to develope the slowly changing dimension 2 

# step 1- first read the data from source 

# as in pyspark foramt 
## suppose you have a source  the data 
df.createOrReplaceTempView("scr_table name")

spark.sql("select * from tempPtable").display()

## Now compare the current data with new data and check if any changes , 
merge into target_table src
using source table a
on a.id=src.id and trg.isactive='Y'
when matched and 
a.name <> src.name or a.email <> src.email 

then update set 
trg.is_active="N",
trg.end_date=current_timestamp()

## this is the first step to update the changes record should be update in final table 
# but we have to also preserve the history , so we need to again run the merge based on the id 

merge into trg g 
using src s on 
trg.id=src.id and trg.is_active="N"

when not matched then * 



## this is the source side

Q2. What you do in slowly changing dimension 1  - 
In slowly changing dimension 1 we just upsert the record, here we are not maintingin any history 

# code implementtaion 
merge into target table trg 
using source table src 
on src.id=trg.id
when matched then update * 
when not matched then insert *

## next question is find the secord highest salary 
# okay what is the next another approach
select max(salary) as second_salary from employee_table 
where salary< (select max(salary) from table employee_table )

## check string are anagram or not



In [ ]:

# Write an SQL query to identify customers whose latest order amount is
higher than their previous order.

with cte as (
    
    select customerid, orderdate, total_amount,
    lag(total_amount) over(partition by customerid order by orderdate) previous,
    row_number() over(partition by customer_id order by desc) rn 
    from orders 
)
    select customerid, orderdate, total amount, previos
    where rn=1 and total_amount>previous


In [ ]:
# Write an SQL query to calculate the average gap between consecutive purchases for each customer.

with cte as (
    select customerid, orderdate ,
    lag(orderdate) over (partition by customerid order by orderdate) previous 
    from orders
)
select customerid, orderdate, previos, avg(orderdate-previos) as avg_gap
from cte 
where previos_date is not null 
group by customerid

## another solution is but with this solution how can you calculate the avg gap 
select customer_id, avg(orderdate-previos) as gag from (
    select customer_id , orderdate, 
    lag(orderdate) over(partition by customerid order by orderdate) previous
    from orders 
)
where previous is not null 
group by customerid;



In [ ]:
# Write an SQL query to calculate year-over-year revenue growth

select * from (select extract(year from date) as year ,
 sum(total_amount) as revenue,
 revenue-lag(revenue) over (order by extrac(year from date)) rv
 from orders
 ) t ;


In [ ]:
#Write an SQL query to identify which day of the week generates the highest
select extract(DOW from orderdate) as  week, sum(total_amount) as revenue
from sales
group by extract(DOW from orderdate) as weeek
order by revenue desc
limit 1;


In [ ]:
# Write an SQL query to identify customers who did not place any repeat 
# order within their first 30 days after signup
select c1.customerid,
o1.orderid
from customers c1 
left join orders o1 
on c1.customerid=o1.customerid and 
o1.orderdate<=c1.signumdate+interval '30 days'
group by c1.customerid
having count(o.orderid)=0;




In [ ]:
# Find the customers who made the order after login, 

""" select DISTINCT l.customerid,p.event_timestamp from customer_event l  
join customer_event p 
on l.customerid=p.customerid
WHERE l.event_type='login' and p.event_type='order_purchased'
and p.event_timestamp>l.event_timestamp;


-- another approach 

WITH ordered_events AS (
    SELECT
        customerid,
        event_type,
        event_timestamp,
        LAG(event_type) OVER (
            PARTITION BY customerid
            ORDER BY event_timestamp
        ) AS previous_event
    FROM customer_event
)
SELECT customerid,event_timestamp
FROM ordered_events
WHERE event_type = 'order_purchased'
  AND previous_event = 'login';
  

  --third approach 
  
  SELECT DISTINCT c.customerid
FROM customer_event c
WHERE c.event_type = 'login'
  AND EXISTS (
      SELECT 1
      FROM customer_event o
      WHERE o.customerid = c.customerid
        AND o.event_type = 'order_purchased'
        AND o.event_timestamp > c.event_timestamp
  );
  
  
  
  """